# 2.10 对比学习 (Contrastive Learning)

通过拉近相似样本、推远不相似样本的表征来学习有意义的特征表示。对比学习是CLIP等跨模态模型和嵌入模型训练的核心方法。

本节涵盖：
- 自监督对比学习
- 监督对比学习
- 跨模态对比学习
- 难负样本挖掘

## 1. 自监督对比学习

**目的**：无需标签学习通用表征

**基本原理**：对同一输入进行两次不同的数据增强得到正样本对，不同输入互为负样本对，使用InfoNCE损失拉近正对、推远负对。代表方法：SimCLR、MoCo。

**InfoNCE损失**：L = -log(exp(sim(z_i, z_j)/τ) / Σ exp(sim(z_k, z_l)/τ))

其中τ是温度参数，sim是余弦相似度。分子是正对的相似度，分母是所有对的相似度之和。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ContrastiveModel(nn.Module):
    def __init__(self, input_dim=64, hidden=128, proj_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.projector = nn.Sequential(
            nn.Linear(hidden, proj_dim), nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
    def forward(self, x):
        return F.normalize(self.projector(self.encoder(x)), dim=-1)

def info_nce_loss(z1, z2, temperature=0.07):
    batch = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / temperature
    labels = torch.cat([torch.arange(batch, 2*batch), torch.arange(0, batch)])
    mask = torch.eye(2*batch, dtype=torch.bool)
    sim.masked_fill_(mask, -1e9)
    return F.cross_entropy(sim, labels)

model = ContrastiveModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_samples = 200
X = torch.randn(n_samples, 64)

print('=== Self-Supervised Contrastive Learning (SimCLR-style) ===')
for epoch in range(30):
    x1 = X + torch.randn_like(X) * 0.1
    x2 = X + torch.randn_like(X) * 0.1
    z1, z2 = model(x1), model(x2)
    loss = info_nce_loss(z1, z2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        pos_sim = F.cosine_similarity(z1, z2).mean()
        neg_sim = F.cosine_similarity(z1[:50], z2[50:100]).mean()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, pos_sim={pos_sim:.3f}, neg_sim={neg_sim:.3f}')

print('\nKey: Positive pairs (augmentations of same input) are pulled together.')
print('Negative pairs (different inputs) are pushed apart.')

## 2. 监督对比学习

**目的**：利用标签信息构建更精确的正负对

**基本原理**：同一类的样本互为正对，不同类的样本互为负对，比自监督对比学习能学到更符合语义的表征。

**与自监督对比的区别**：
- 自监督：正对=同一输入的两次增强，负对=不同输入
- 监督：正对=同类样本，负对=不同类样本
- 监督对比可以有多个正对（同类的所有样本）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def supervised_contrastive_loss(features, labels, temperature=0.07):
    device = features.device
    n = features.size(0)
    sim = torch.mm(features, features.t()) / temperature
    label_mask = labels.unsqueeze(0) == labels.unsqueeze(1)
    diag_mask = ~torch.eye(n, dtype=torch.bool, device=device)
    positive_mask = label_mask & diag_mask
    valid_mask = positive_mask.sum(dim=1) > 0

    if not valid_mask.any():
        return torch.tensor(0.0, device=device, requires_grad=True)

    sim_exp = torch.exp(sim)
    sim_exp = sim_exp.masked_fill(~diag_mask, 0)
    log_prob = sim - torch.log(sim_exp.sum(dim=1, keepdim=True) + 1e-8)
    mean_log_prob = (log_prob * positive_mask.float()).sum(dim=1) / (positive_mask.float().sum(dim=1) + 1e-8)
    loss = -mean_log_prob[valid_mask].mean()
    return loss

class SupConModel(nn.Module):
    def __init__(self, input_dim=32, hidden=64, proj_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.projector = nn.Sequential(nn.Linear(hidden, proj_dim), nn.Linear(proj_dim, proj_dim))
    def forward(self, x):
        return F.normalize(self.projector(self.encoder(x)), dim=-1)

model = SupConModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_classes = 4
n_per_class = 30
X = torch.cat([torch.randn(n_per_class, 32) + c * 2 for c in range(n_classes)])
labels = torch.cat([torch.full((n_per_class,), c, dtype=torch.long) for c in range(n_classes)])

print('=== Supervised Contrastive Learning ===')
for epoch in range(30):
    features = model(X)
    loss = supervised_contrastive_loss(features, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        with torch.no_grad():
            intra_sim = sum(F.cosine_similarity(
                features[labels==c].mean(0, keepdim=True).expand(n_per_class, -1),
                features[labels==c]
            ).mean() for c in range(n_classes)) / n_classes
            inter_sims = []
            centroids = [features[labels==c].mean(0) for c in range(n_classes)]
            for i in range(n_classes):
                for j in range(i+1, n_classes):
                    inter_sims.append(F.cosine_similarity(centroids[i].unsqueeze(0), centroids[j].unsqueeze(0)).item())
            inter_sim = sum(inter_sims) / len(inter_sims)
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, intra_sim={intra_sim:.3f}, inter_sim={inter_sim:.3f}')

print('\nKey: Same-class samples cluster together, different-class samples separate.')

## 3. 跨模态对比学习（CLIP风格）

**目的**：对齐不同模态的表征空间

**基本原理**：将匹配的跨模态对（如图文对）作为正对，不匹配的作为负对，使用对比损失拉近匹配对、推远不匹配对。代表：CLIP在4亿图文对上训练，实现零样本图像分类。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ImageEncoder(nn.Module):
    def __init__(self, input_dim=64, proj_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, proj_dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

class TextEncoder(nn.Module):
    def __init__(self, input_dim=32, proj_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, proj_dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def clip_loss(image_features, text_features, temperature=0.07):
    logits = image_features @ text_features.t() / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    return (loss_i2t + loss_t2i) / 2

img_enc = ImageEncoder()
txt_enc = TextEncoder()
optimizer = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()), lr=1e-3)

n_pairs = 64
image_data = torch.randn(n_pairs, 64)
text_data = torch.randn(n_pairs, 32)
for i in range(n_pairs):
    text_data[i] = image_data[i, :32] + torch.randn(32) * 0.3

print('=== Cross-Modal Contrastive Learning (CLIP-style) ===')
for epoch in range(30):
    img_feat = img_enc(image_data)
    txt_feat = txt_enc(text_data)
    loss = clip_loss(img_feat, txt_feat)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        with torch.no_grad():
            sim = img_feat @ txt_feat.t()
            r1 = (sim.argmax(dim=1) == torch.arange(n_pairs)).float().mean()
            r5 = sum((sim.argsort(dim=1, descending=True)[:, :5] == torch.arange(n_pairs).unsqueeze(1)).any(dim=1).float()) / n_pairs
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, R@1={r1:.3f}, R@5={r5:.3f}')

print('\nKey: CLIP aligns image and text embeddings via bidirectional contrastive loss.')
print('R@1 = Recall@1: correct match is top-1 result.')

## 4. 难负样本挖掘

**目的**：提升对比学习的判别能力

**基本原理**：自动选择与锚点相似但不匹配的难负样本参与对比损失计算，使模型学到更精细的判别边界，在检索和嵌入模型训练中至关重要。

**为什么难负样本重要**：
- 随机负样本通常与锚点差异很大，模型容易区分，提供的学习信号弱
- 难负样本与锚点相似但不匹配，迫使模型学习更精细的区分能力
- 代表：BGE、E5等嵌入模型训练中使用难负样本挖掘

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class HardNegativeMiner:
    def __init__(self, n_hard_negatives=3):
        self.n_hard_negatives = n_hard_negatives

    def mine(self, query_features, corpus_features, query_labels, corpus_labels):
        sim = query_features @ corpus_features.t()
        label_match = query_labels.unsqueeze(1) == corpus_labels.unsqueeze(0)
        sim[label_match] = -2.0
        sim.fill_diagonal_(-2.0)
        hard_neg_indices = sim.argsort(dim=-1, descending=True)[:, :self.n_hard_negatives]
        return hard_neg_indices

def hard_negative_loss(query_feat, pos_feat, hard_neg_feats, temperature=0.07):
    pos_sim = F.cosine_similarity(query_feat, pos_feat, dim=-1) / temperature
    neg_sims = []
    for neg_feat in hard_neg_feats:
        neg_sims.append(F.cosine_similarity(query_feat, neg_feat, dim=-1) / temperature)
    neg_sim = torch.stack(neg_sims, dim=-1)
    logits = torch.cat([pos_sim.unsqueeze(-1), neg_sim], dim=-1)
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, labels)

n_samples = 100
n_classes = 5
labels = torch.randint(0, n_classes, (n_samples,))
features = torch.randn(n_samples, 32)
for c in range(n_classes):
    mask = labels == c
    features[mask] += c * 2
features = F.normalize(features, dim=-1)

miner = HardNegativeMiner(n_hard_negatives=3)

n_queries = 20
query_idx = torch.randperm(n_samples)[:n_queries]
hard_negs = miner.mine(features[query_idx], features, labels[query_idx], labels)

print('=== Hard Negative Mining Demo ===')
for i in range(5):
    qi = query_idx[i]
    ql = labels[qi].item()
    neg_labels = [labels[hard_negs[i, j]].item() for j in range(3)]
    neg_sims = [F.cosine_similarity(features[qi].unsqueeze(0), features[hard_negs[i, j]].unsqueeze(0)).item() for j in range(3)]
    pos_sims = [F.cosine_similarity(features[qi].unsqueeze(0), features[labels == ql][j].unsqueeze(0)).item() for j in range(min(2, (labels == ql).sum().item() - 1))]
    print(f'Query {qi} (class {ql}):')
    print(f'  Hard negatives: classes={neg_labels}, sims={[f"{s:.3f}" for s in neg_sims]}')
    print(f'  Positives: sims={[f"{s:.3f}" for s in pos_sims]}')

print('\nKey: Hard negatives are similar to the query but from different classes.')
print('Training on hard negatives forces the model to learn fine-grained distinctions.')